# SIFLOW · run_4 · MDLM ablations (fills Table 3)

Retrains short (5k-step) head variants and evaluates each at k=1,8. ~5–7h total; each variant resumes independently if interrupted.

**Runtime:** designed to finish well under one Colab session. Training stops and checkpoints automatically at an 11h guard — if that happens, just re-run this notebook (re-import its checkpoint) and it resumes.

**How to use:** run every cell top-to-bottom. Cells 1–2 set up the environment and the artifact location; the last cell downloads this part's output zip for the next notebook.

In [ ]:
# === 1. Clone + install (run once per session, ~2 min) ===
REPO_URL = "https://github.com/kagtgi/siflow.git"
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# === 2. Where do artifacts live? ===
# Default: a local folder + zip download/upload between parts (no Drive needed).
# Set USE_DRIVE = True to persist on Google Drive instead (then the import and
# download steps below become no-ops).
USE_DRIVE = False

import os
if USE_DRIVE:
    from siflow.utils import drive
    drive.mount()
    os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
    BASE = drive.base_dir()
else:
    BASE = "/content/artifacts"
    os.makedirs(BASE, exist_ok=True)
print("artifacts ->", BASE)

### Import the previous part(s)

This part needs the output zip(s) you downloaded earlier. Run the cell below — a file picker opens; select **all** of these at once:

- `run_1_data.zip` — tokenized data + val
- `run_3_results.zip` — (optional) keeps run_3's main-table rows alongside the ablations

*(If a long run here stopped early at the 11h guard, also re-upload **this** part's own checkpoint zip to resume.)* Using Drive instead? Set `USE_DRIVE=True` above and skip this.

In [ ]:
# === Import previous outputs (pick the .zip files listed above) ===
if not USE_DRIVE:
    from siflow.utils.io import import_zips
    import_zips(BASE)
else:
    print('USE_DRIVE: reading prior outputs from', BASE)

In [ ]:
ABLATIONS = {
  "abl_no_avg_velocity": "ablation.no_avg_velocity=true",
  "abl_hard_label":      "ablation.hard_label=true",
  "abl_no_entropy_prior":"train.lam_ent=0.0",
  "abl_identity_target": "train.w_id=0.5",
  "abl_prob_space":      "head.space=prob",
  "abl_head_depth1":     "head.mid_blocks=1",
}
for rid, override in ABLATIONS.items():
    print("=== train", rid, "===")
    !python scripts/train.py --config siflow/config/mdlm.yaml --set \
        data.tokens_path={BASE}/data/owt_gpt2_256.npy out_dir={BASE}/ckpt/{rid} \
        run_id={rid} train.total_steps=5000 train.max_hours=1.5 {override}
    !python scripts/evaluate.py --config siflow/config/mdlm.yaml --system siflow \
        --ckpt-dir {BASE}/ckpt/{rid} --ref-tokens {BASE}/data/owt_gpt2_val.npy \
        --n-samples 500 --k-list 1 8 --set run_id={rid} --out {BASE}/results/{rid}.json

In [ ]:
!python scripts/make_tables.py --results {BASE}/results --out {BASE}/tables_auto.tex
print(open(f'{BASE}/tables_auto.tex').read())

In [ ]:
# === Save + auto-download this part's output ===
from siflow.utils.io import export_and_download
if not USE_DRIVE:
    export_and_download(BASE, 'run_4_ablations.zip', include=['results', 'tables_auto.tex'])
else:
    print('USE_DRIVE: outputs already persisted under', BASE)